In [3]:
import pandas as pd
import numpy as np
from datetime import datetime

# Load the dataset
# Note: Using semicolon as delimiter and specifying decimal comma
df = pd.read_csv('../data/raw/AirQualityUCI.csv', sep=';', decimal=',')

print("Original dataset shape:", df.shape)
print("\nOriginal columns:", df.columns.tolist())

# Remove empty columns (columns with all NaN or empty strings)
df = df.dropna(axis=1, how='all')
df = df.loc[:, (df != '').any(axis=0)]

print("\nAfter removing empty columns:", df.shape)

# Clean column names
column_mapping = {
    'CO(GT)': 'CO',
    'PT08.S1(CO)': 'PT08_S1_CO',
    'NMHC(GT)': 'NMHC',
    'C6H6(GT)': 'C6H6',
    'PT08.S2(NMHC)': 'PT08_S2_NMHC',
    'NOx(GT)': 'NOx',
    'PT08.S3(NOx)': 'PT08_S3_NOx',
    'NO2(GT)': 'NO2',
    'PT08.S4(NO2)': 'PT08_S4_NO2',
    'PT08.S5(O3)': 'PT08_S5_O3'
}
df = df.rename(columns=column_mapping)

# Combine Date and Time into DateTime
df['DateTime'] = pd.to_datetime(
    df['Date'] + ' ' + df['Time'], 
    format='%d/%m/%Y %H.%M.%S',
    errors='coerce'
)

# Drop original Date and Time columns
df = df.drop(['Date', 'Time'], axis=1)

# Reorder columns to put DateTime first
cols = ['DateTime'] + [col for col in df.columns if col != 'DateTime']
df = df[cols]

# Replace -200 with NaN (missing value indicator)
pollutant_columns = [col for col in df.columns if col != 'DateTime']
df[pollutant_columns] = df[pollutant_columns].replace(-200, np.nan)
df[pollutant_columns] = df[pollutant_columns].replace(-200.0, np.nan)

# Remove rows with invalid DateTime
initial_rows = len(df)
df = df.dropna(subset=['DateTime'])
print(f"\nRows removed due to invalid DateTime: {initial_rows - len(df)}")

# Remove rows where ALL pollutant measurements are missing
df_before = len(df)
df = df.dropna(subset=pollutant_columns, how='all')
rows_all_missing = df_before - len(df)
print(f"Rows removed where ALL pollutants are missing: {rows_all_missing}")

# Calculate statistics
total_rows = len(df)
rows_with_missing = df[pollutant_columns].isna().any(axis=1).sum()
rows_complete = total_rows - rows_with_missing

print("\n" + "="*60)
print("CLEANING SUMMARY")
print("="*60)
print(f"Final dataset shape: {df.shape}")
print(f"Total rows retained: {total_rows}")
print(f"Rows with complete data: {rows_complete}")
print(f"Rows with some missing values: {rows_with_missing}")
print(f"Completeness: {(rows_complete / total_rows * 100):.1f}%")

# Show missing value statistics per column
print("\nMissing values per column:")
missing_stats = df[pollutant_columns].isna().sum().sort_values(ascending=False)
for col, count in missing_stats.items():
    if count > 0:
        pct = (count / total_rows) * 100
        print(f"  {col}: {count} ({pct:.1f}%)")

# Display first few rows
print("\nFirst 5 rows of cleaned data:")
print(df.head())

# Display data types
print("\nData types:")
print(df.dtypes)

# Save cleaned dataset
output_filename = 'AirQualityUCI_cleaned.csv'
df.to_csv(output_filename, index=False)
print(f"\n✓ Cleaned dataset saved to: {output_filename}")

# Optional: Additional time-based features for HMM analysis
print("\n" + "="*60)
print("EXTRACTING TIME FEATURES (Optional for HMM)")
print("="*60)

df['Hour'] = df['DateTime'].dt.hour
df['DayOfWeek'] = df['DateTime'].dt.dayofweek  # 0=Monday, 6=Sunday
df['Month'] = df['DateTime'].dt.month
df['DayOfYear'] = df['DateTime'].dt.dayofyear

# Save version with time features
output_with_features = 'AirQualityUCI_cleaned_with_time_features.csv'
df.to_csv(output_with_features, index=False)
print(f"✓ Dataset with time features saved to: {output_with_features}")

print("\nTime feature ranges:")
print(f"  Date range: {df['DateTime'].min()} to {df['DateTime'].max()}")
print(f"  Hours: {df['Hour'].min()} to {df['Hour'].max()}")
print(f"  Months: {df['Month'].unique()}")

print("\n" + "="*60)
print("NEXT STEPS FOR HMM ANALYSIS")
print("="*60)
print("1. Handle remaining missing values (interpolation, forward fill, etc.)")
print("2. Select relevant features for HMM (e.g., CO, NOx, NO2, C6H6)")
print("3. Normalize/standardize features")
print("4. Train HMM with different numbers of hidden states (2-5 regimes)")
print("5. Analyze regime transitions by time of day, day of week, season")
print("="*60)

Original dataset shape: (9471, 17)

Original columns: ['Date', 'Time', 'CO(GT)', 'PT08.S1(CO)', 'NMHC(GT)', 'C6H6(GT)', 'PT08.S2(NMHC)', 'NOx(GT)', 'PT08.S3(NOx)', 'NO2(GT)', 'PT08.S4(NO2)', 'PT08.S5(O3)', 'T', 'RH', 'AH', 'Unnamed: 15', 'Unnamed: 16']

After removing empty columns: (9471, 15)

Rows removed due to invalid DateTime: 114
Rows removed where ALL pollutants are missing: 31

CLEANING SUMMARY
Final dataset shape: (9326, 14)
Total rows retained: 9326
Rows with complete data: 827
Rows with some missing values: 8499
Completeness: 8.9%

Missing values per column:
  NMHC: 8412 (90.2%)
  CO: 1652 (17.7%)
  NO2: 1611 (17.3%)
  NOx: 1608 (17.2%)
  PT08_S1_CO: 335 (3.6%)
  C6H6: 335 (3.6%)
  PT08_S2_NMHC: 335 (3.6%)
  PT08_S3_NOx: 335 (3.6%)
  PT08_S4_NO2: 335 (3.6%)
  PT08_S5_O3: 335 (3.6%)
  T: 335 (3.6%)
  RH: 335 (3.6%)
  AH: 335 (3.6%)

First 5 rows of cleaned data:
             DateTime   CO  PT08_S1_CO   NMHC  C6H6  PT08_S2_NMHC    NOx  \
0 2004-03-10 18:00:00  2.6      1360.0 

In [4]:
#####################################################################################################################
#  Filtering out the cleaned dataset
#
#####################################################################################################################

In [7]:
import pandas as pd
import numpy as np

# Load the cleaned dataset
df = pd.read_csv('AirQualityUCI_cleaned.csv')

print("Original cleaned dataset shape:", df.shape)
print("Original columns:", df.columns.tolist())

# Define the variables to keep for HMM analysis
variables_to_keep = ['DateTime', 'NOx', 'NO2', 'CO', 'C6H6', 'T']

# Check which variables exist in the dataset
available_vars = [var for var in variables_to_keep if var in df.columns]
missing_vars = [var for var in variables_to_keep if var not in df.columns]

if missing_vars:
    print(f"\nWarning: The following variables are not in the dataset: {missing_vars}")
    print("Available columns:", df.columns.tolist())

# Filter dataset to keep only specified variables
df_filtered = df[available_vars].copy()

print("\n" + "="*60)
print("FILTERING TO HMM VARIABLES")
print("="*60)
print(f"Variables selected: {available_vars}")
print(f"Filtered dataset shape: {df_filtered.shape}")

# Convert DateTime to datetime format if it's not already
df_filtered['DateTime'] = pd.to_datetime(df_filtered['DateTime'])

# Analyze missing values in the filtered dataset
print("\n" + "="*60)
print("MISSING VALUE ANALYSIS")
print("="*60)

pollutant_cols = [col for col in available_vars if col != 'DateTime']
total_rows = len(df_filtered)

# Missing values per column
print("\nMissing values per variable:")
for col in pollutant_cols:
    missing_count = df_filtered[col].isna().sum()
    missing_pct = (missing_count / total_rows) * 100
    print(f"  {col}: {missing_count} ({missing_pct:.1f}%)")

# Rows with any missing values
rows_with_missing = df_filtered[pollutant_cols].isna().any(axis=1).sum()
rows_complete = total_rows - rows_with_missing

print(f"\nTotal rows: {total_rows}")
print(f"Rows with complete data (no missing): {rows_complete} ({rows_complete/total_rows*100:.1f}%)")
print(f"Rows with some missing values: {rows_with_missing} ({rows_with_missing/total_rows*100:.1f}%)")

# Option 1: Keep all rows (with missing values)
print("\n" + "="*60)
print("OPTION 1: KEEP ALL ROWS (Handle missing values later)")
print("="*60)
output_all = 'AirQualityUCI_HMM_variables_all_rows.csv'
df_filtered.to_csv(output_all, index=False)
print(f"✓ Saved: {output_all}")
print(f"  Shape: {df_filtered.shape}")

# Option 2: Keep only complete cases (no missing values)
print("\n" + "="*60)
print("OPTION 2: COMPLETE CASES ONLY (No missing values)")
print("="*60)
df_complete = df_filtered.dropna(subset=pollutant_cols)
output_complete = 'AirQualityUCI_HMM_variables_complete_cases.csv'
df_complete.to_csv(output_complete, index=False)
print(f"✓ Saved: {output_complete}")
print(f"  Shape: {df_complete.shape}")
print(f"  Data loss: {(1 - len(df_complete)/total_rows)*100:.1f}%")

# Display sample of data
print("\n" + "="*60)
print("DATA PREVIEW (First 10 rows)")
print("="*60)
print(df_filtered.head(10))

# Display basic statistics
print("\n" + "="*60)
print("DESCRIPTIVE STATISTICS (All rows)")
print("="*60)
print(df_filtered[pollutant_cols].describe())

# Check for data range and potential outliers
print("\n" + "="*60)
print("DATA RANGE CHECK")
print("="*60)
for col in pollutant_cols:
    valid_data = df_filtered[col].dropna()
    if len(valid_data) > 0:
        print(f"\n{col}:")
        print(f"  Min: {valid_data.min():.2f}")
        print(f"  Max: {valid_data.max():.2f}")
        print(f"  Mean: {valid_data.mean():.2f}")
        print(f"  Median: {valid_data.median():.2f}")
        print(f"  Std: {valid_data.std():.2f}")

print("\n" + "="*60)
print("NEXT STEPS FOR HMM ANALYSIS")
print("="*60)
print("1. Choose which dataset to use:")
print("   - All rows (handle missing values via interpolation/imputation)")
print("   - Complete cases only (may lose significant data)")
print("\n2. Handle missing values if using 'all rows' version:")
print("   - Forward fill: df.fillna(method='ffill')")
print("   - Interpolation: df.interpolate(method='linear')")
print("   - Mean imputation: df.fillna(df.mean())")
print("\n3. Normalize/standardize the features:")
print("   - StandardScaler or MinMaxScaler from sklearn")
print("\n4. Train HMM model with hmmlearn library")
print("="*60)

Original cleaned dataset shape: (9326, 14)
Original columns: ['DateTime', 'CO', 'PT08_S1_CO', 'NMHC', 'C6H6', 'PT08_S2_NMHC', 'NOx', 'PT08_S3_NOx', 'NO2', 'PT08_S4_NO2', 'PT08_S5_O3', 'T', 'RH', 'AH']

FILTERING TO HMM VARIABLES
Variables selected: ['DateTime', 'NOx', 'NO2', 'CO', 'C6H6', 'T']
Filtered dataset shape: (9326, 6)

MISSING VALUE ANALYSIS

Missing values per variable:
  NOx: 1608 (17.2%)
  NO2: 1611 (17.3%)
  CO: 1652 (17.7%)
  C6H6: 335 (3.6%)
  T: 335 (3.6%)

Total rows: 9326
Rows with complete data (no missing): 6941 (74.4%)
Rows with some missing values: 2385 (25.6%)

OPTION 1: KEEP ALL ROWS (Handle missing values later)
✓ Saved: AirQualityUCI_HMM_variables_all_rows.csv
  Shape: (9326, 6)

OPTION 2: COMPLETE CASES ONLY (No missing values)
✓ Saved: AirQualityUCI_HMM_variables_complete_cases.csv
  Shape: (6941, 6)
  Data loss: 25.6%

DATA PREVIEW (First 10 rows)
             DateTime    NOx    NO2   CO  C6H6     T
0 2004-03-10 18:00:00  166.0  113.0  2.6  11.9  13.6
1 200